# Bag-of-Words & TF-IDF — documents as vectors

> Tutorial pair for [`bow_tfidf.py`](bow_tfidf.py).

## 1. Intuition
Before we can do any math on text we need numbers. The simplest idea: throw the
words of a document into a "bag", forget their order, and count them. That is
**Bag-of-Words**. But raw counts over-reward words like *"the"* that appear
everywhere and carry no topical signal. **TF-IDF** fixes this by multiplying each
count by how *rare* the word is across the whole corpus — so distinctive words
dominate the vector. Comparing the resulting vectors with cosine similarity is
the bread-and-butter of classic search engines.

## 2. Concept (the slide)
- **Vocabulary:** the sorted set of all terms; column $t$ of the matrix.
- **Term frequency** $\mathrm{tf}(t,d)$: how often term $t$ appears in document $d$.
- **Document frequency** $\mathrm{df}(t)$: in how many documents $t$ appears at all.
- **IDF** down-weights common terms: $\log\frac{N}{\mathrm{df}(t)}$ (smoothed).
- **TF-IDF** $=\mathrm{tf}\cdot\mathrm{idf}$, then **L2-normalize** each row so
  document length doesn't matter.
- **Cosine similarity** measures the angle between two document vectors — robust
  to length, unlike raw dot products.

## 3. Math derivation

**Term frequency.** The base signal is the raw count
$$\mathrm{tf}(t,d)=\operatorname{count}(t,d).$$
A document that mentions "market" five times is more about markets than one that
mentions it once — but probably not *five times* more. The **sublinear** variant
dampens this:
$$\mathrm{tf}(t,d)=1+\log\operatorname{count}(t,d)\quad(\text{when count}>0).$$

**Inverse document frequency.** A term in *every* document discriminates nothing.
Let $N$ be the number of documents and $\mathrm{df}(t)$ the number containing $t$.
The unsmoothed IDF is
$$\mathrm{idf}(t)=\log\frac{N}{\mathrm{df}(t)}.$$
To avoid division by zero (a term never seen at fit time) and never-negative
weights, we use the **smoothed** form (the sklearn default), which pretends an
extra document contains every term:
$$\boxed{\;\mathrm{idf}(t)=\log\frac{1+N}{1+\mathrm{df}(t)}+1\;}$$
The trailing $+1$ keeps even ubiquitous terms ($\mathrm{df}=N$, so the log is
$\log\frac{1+N}{1+N}=0$) from being zeroed out entirely.

**TF-IDF weighting.**
$$w(t,d)=\mathrm{tf}(t,d)\,\cdot\,\mathrm{idf}(t).$$

**L2 normalization.** Long documents have larger counts everywhere, inflating all
weights. We project each row onto the unit sphere:
$$\hat w_d=\frac{w_d}{\lVert w_d\rVert_2},\qquad \lVert w_d\rVert_2=\sqrt{\textstyle\sum_t w(t,d)^2}.$$

**Cosine similarity.** The similarity of documents $a,b$ is the cosine of the
angle between their vectors:
$$\cos(a,b)=\frac{a\cdot b}{\lVert a\rVert\,\lVert b\rVert}.$$
After L2 normalization $\lVert a\rVert=\lVert b\rVert=1$, so this collapses to the
plain dot product $\hat a\cdot\hat b$ — fast to compute for retrieval. To rank
documents for a query $q$ we vectorize $q$ the same way and sort by
$\cos(q,\,d)$.

## 4. NumPy implementation

In [ ]:
# ===== actual implementation from bow_tfidf.py =====
from __future__ import annotations

import re

from collections import Counter

import numpy as np

SEED = 0

_TOKEN_RE = re.compile(r"[a-z]+")

def tokenize(text: str) -> list[str]:
    """Lowercase and split into alphabetic tokens (the simplest sane tokenizer)."""
    return _TOKEN_RE.findall(text.lower())

def sklearn_tfidf(docs: list[str]):
    """Return (matrix, feature_names) from sklearn for cross-checking."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    # token_pattern matches our tokenizer: lowercase alphabetic runs
    v = TfidfVectorizer(token_pattern=r"[a-z]+", lowercase=True)
    X = v.fit_transform(docs).toarray()
    return X, list(v.get_feature_names_out())

def toy_corpus() -> list[str]:
    return [
        "the cat sat on the mat",
        "the dog sat on the log",
        "cats and dogs are popular pets",
        "the stock market fell sharply today",
        "investors sold stocks as the market dropped",
        "a pet dog loves to play and run",
    ]

def demo():
    np.random.seed(SEED)
    docs = toy_corpus()

    bow = CountVectorizerNumPy()
    X = bow.fit_transform(docs)
    print(f"BoW matrix shape = {X.shape} (docs x vocab); vocab size = {len(bow.feature_names_)}")
    print(f"counts for doc 0 nonzero terms: "
          f"{[(bow.feature_names_[j], int(X[0, j])) for j in np.nonzero(X[0])[0]]}")

    tfidf = TfidfVectorizerNumPy()
    T = tfidf.fit_transform(docs)
    # highest-idf (most discriminative) terms
    top_idf = np.argsort(-tfidf.idf_)[:5]
    print(f"\nMost discriminative terms (highest idf): "
          f"{[tfidf.feature_names_[j] for j in top_idf]}")

    # retrieval
    query = "pets like cats and dogs"
    print(f"\nQuery: {query!r}")
    for rank, (i, s) in enumerate(retrieve(query, docs, tfidf, k=3), 1):
        print(f"  {rank}. (sim={s:.3f}) {docs[i]!r}")

    # cross-check our TF-IDF against sklearn
    try:
        Xs, names = sklearn_tfidf(docs)
        # align columns: our feature order is sorted, so is sklearn's
        same_vocab = names == tfidf.feature_names_
        close = np.allclose(T, Xs, atol=1e-6) if same_vocab else False
        print(f"\nMatches sklearn TfidfVectorizer: {close} "
              f"(max abs diff = {np.abs(T - Xs).max():.2e})" if same_vocab
              else "\nVocabularies differ; skipping numeric check.")
    except Exception as e:  # pragma: no cover - sklearn always present here
        print(f"\n(sklearn comparison skipped: {e})")


class CountVectorizerNumPy:
    r"""
    Bag-of-Words: build a vocabulary, then map each document to a count vector.

        X[d, t] = number of times term t occurs in document d   (raw counts)

    With ``binary=True`` we record presence/absence (X in {0, 1}) instead.
    Order is discarded — only multiplicities matter ("bag").
    """

    def __init__(self, binary: bool = False):
        self.binary = binary
        self.vocabulary_: dict[str, int] = {}

    def fit(self, docs: list[str]):
        vocab: dict[str, int] = {}
        for d in docs:
            for w in tokenize(d):
                if w not in vocab:
                    vocab[w] = len(vocab)
        # sort terms for a stable, human-readable column order
        terms = sorted(vocab)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.feature_names_ = terms
        return self

    def transform(self, docs: list[str]) -> np.ndarray:
        X = np.zeros((len(docs), len(self.vocabulary_)))
        for i, d in enumerate(docs):
            for w, c in Counter(tokenize(d)).items():
                j = self.vocabulary_.get(w)
                if j is not None:
                    X[i, j] = 1.0 if self.binary else c
        return X

    def fit_transform(self, docs: list[str]) -> np.ndarray:
        return self.fit(docs).transform(docs)


class TfidfVectorizerNumPy:
    r"""
    TF-IDF from scratch, matching sklearn's default conventions so we can compare.

    **Term frequency** (per document d, term t):
        tf(t, d) = count(t, d)                       # raw, or
        tf(t, d) = 1 + log(count(t, d))  if sublinear and count>0

    **Inverse document frequency** (smoothed, the sklearn default):
        idf(t) = log((1 + N) / (1 + df(t))) + 1
    where N = #documents, df(t) = #documents containing t. The "+1"s are
    smoothing: they prevent zero division and stop idf from ever being negative.
    The trailing "+1" keeps terms that appear in *every* document (idf would be 0)
    from being zeroed out entirely.

    **Weighting:**  tfidf(t, d) = tf(t, d) * idf(t).

    **Normalization:** each document row is L2-normalized,
        x_d <- x_d / ||x_d||_2 ,
    so that the cosine similarity between two documents is just their dot product.
    """

    def __init__(self, sublinear_tf: bool = False, norm: str | None = "l2",
                 smooth_idf: bool = True):
        self.sublinear_tf = sublinear_tf
        self.norm = norm
        self.smooth_idf = smooth_idf

    def fit(self, docs: list[str]):
        # vocabulary (sorted for stable column order)
        vocab: dict[str, int] = {}
        for d in docs:
            for w in set(tokenize(d)):
                vocab[w] = 0
        terms = sorted(vocab)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.feature_names_ = terms

        N = len(docs)
        df = np.zeros(len(terms))
        for d in docs:
            for w in set(tokenize(d)):
                df[self.vocabulary_[w]] += 1.0
        # smoothed idf — the "as if one extra doc contains every term" trick
        s = 1.0 if self.smooth_idf else 0.0
        self.idf_ = np.log((N + s) / (df + s)) + 1.0
        return self

    def _tf_matrix(self, docs: list[str]) -> np.ndarray:
        X = np.zeros((len(docs), len(self.vocabulary_)))
        for i, d in enumerate(docs):
            for w, c in Counter(tokenize(d)).items():
                j = self.vocabulary_.get(w)
                if j is None:
                    continue
                X[i, j] = (1.0 + np.log(c)) if self.sublinear_tf else float(c)
        return X

    def transform(self, docs: list[str]) -> np.ndarray:
        X = self._tf_matrix(docs)
        X = X * self.idf_                      # broadcast idf across columns
        if self.norm == "l2":
            norms = np.linalg.norm(X, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            X = X / norms
        return X

    def fit_transform(self, docs: list[str]) -> np.ndarray:
        return self.fit(docs).transform(docs)

## 5. Reference — cosine retrieval & cross-check against sklearn

In [ ]:
# ===== actual implementation from bow_tfidf.py =====
def cosine_similarity(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    r"""
    Cosine similarity matrix between rows of A and rows of B:
        cos(a, b) = (a . b) / (||a|| ||b||).
    For already-L2-normalized vectors this reduces to the plain dot product.
    """
    A = np.atleast_2d(A)
    B = np.atleast_2d(B)
    an = np.linalg.norm(A, axis=1, keepdims=True)
    bn = np.linalg.norm(B, axis=1, keepdims=True)
    an[an == 0] = 1.0
    bn[bn == 0] = 1.0
    return (A / an) @ (B / bn).T


def retrieve(query: str, docs: list[str], vec: TfidfVectorizerNumPy,
             k: int = 3) -> list[tuple[int, float]]:
    """Rank documents by cosine similarity of their TF-IDF vectors to the query."""
    D = vec.transform(docs)
    q = vec.transform([query])
    sims = cosine_similarity(q, D)[0]
    order = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in order]

## 6. Train / run — build vectors, retrieve, and match sklearn

In [ ]:
demo()

## 7. Visualization — the TF-IDF document–term heatmap

In [ ]:
import matplotlib
matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import bow_tfidf as M

docs = M.toy_corpus()
vec = M.TfidfVectorizerNumPy()
T = vec.fit_transform(docs)
terms = vec.feature_names_

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(T, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(terms))); ax.set_xticklabels(terms, rotation=90, fontsize=7)
ax.set_yticks(range(len(docs))); ax.set_yticklabels([f"doc {i}" for i in range(len(docs))])
ax.set_title("TF-IDF weights (rows=documents, columns=terms)")
fig.colorbar(im, ax=ax, label="tf-idf weight")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- TF-IDF is a sparse, interpretable, *order-free* representation — a strong,
  fast baseline for retrieval and text classification.
- It cannot capture **word meaning or order**: "dog bites man" and "man bites
  dog" get identical vectors, and *cat*/*kitten* are as unrelated as *cat*/*car*.
  Dense [embeddings](../embeddings/word2vec.ipynb) fix the meaning problem;
  sequence models fix the order problem.
- Watch the **smoothing** and **normalization** conventions — they are exactly
  why our from-scratch matrix reproduces sklearn's to machine precision.
- IDF must be fit on the **training corpus only**; reuse the same `idf_` to
  transform held-out queries/documents.